<a id="top"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Aptos;text-align:center;display:fill;border-radius:5px;background-color:#00008B;overflow:hidden"><b>Actuarial Agents: Responsible AI Agent Teams for Actuarial Risk Prediction</b></div>

<div style="padding:16px;color:darkblue;margin:0;font-size:18px;font-family:Aptos;text-align:left;display:fill;border-radius:5px;background-color:white;overflow:hidden"><b>Author: Bart Custers</b></div>

<div style="padding:16px;color:darkblue;margin:0;font-size:18px;font-family:Aptos;text-align:left;display:fill;border-radius:5px;background-color:white;overflow:hidden"><i>MSc Thesis Artificial Intelligence, School of Computer Science, University of Hull</i></div>

<a id="top"></a>
# <div style="padding:20px;color:white;margin:0;font-size:24px;font-family:Aptos;text-align:left;display:fill;border-radius:5px;background-color:#00008B;overflow:hidden"><b>Testing: Notebook for deriving test datasets to test and evaluate the multi agent framework.</b></div>

### **Import libraries and data**

In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

In [2]:
# Load the dataset (replace with your actual file path)
df_base = pd.read_csv('C:/Users/bart_/Documents/git_repo/agentic_actuaries/data/raw/freMTPL2freq.csv')

# Inspect the dataset
df_base.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
0,1.0,1,0.10,D,5,0,55,50,B12,Regular,1217,R82
1,3.0,1,0.77,D,5,0,55,50,B12,Regular,1217,R82
2,5.0,1,0.75,B,6,2,52,50,B12,Diesel,54,R22
3,10.0,1,0.09,B,7,0,46,50,B12,Diesel,76,R72
4,11.0,1,0.84,B,7,0,46,50,B12,Diesel,76,R72


### **1. Missing values**

In [3]:
# Randomly introduce missing values in the 'DrivAge' column for 10% of the rows
df_T1 = df_base.copy()

mask = rng.random(len(df_T1)) < 0.1
df_T1.loc[mask, "DrivAge"] = np.nan

In [4]:
# Randomly introduce missing values in the 'BonusMalus' column for 50% of the rows
df_T2 = df_base.copy()

mask = rng.random(len(df_T2)) < 0.5
df_T2.loc[mask, "BonusMalus"] = np.nan

### **2. Missing rows**

In [5]:
# Randomly remove 20,000 rows from the dataset
df_T3 = df_base.copy()

drop_idx = rng.choice(df_T3.index, size=20000, replace=False)
df_T3 = df_T3.drop(index=drop_idx).reset_index(drop=True)

In [6]:
# Randomly remove 100,000 rows from the dataset
df_T4 = df_base.copy()

drop_idx = rng.choice(df_T4.index, size=100000, replace=False)
df_T4 = df_T4.drop(index=drop_idx).reset_index(drop=True)

### **3. Missing columns**

In [7]:
# Remove the 'VehAge' column from the dataset
df_T5 = df_base.copy()

df_T5 = df_T5.drop(columns=["VehAge"])

### **4. Data increase**

In [8]:
# Add 5% extra data to the dataset by bootstrapping (sampling with replacement)
df_T6 = df_base.copy()

increase_frac = 0.05
n_new = int(len(df_T6) * increase_frac)

bootstrap_sample = df_T6.sample(
    n=n_new,
    replace=True,
    random_state=RANDOM_STATE
)

df_T6 = pd.concat([df_T6, bootstrap_sample], ignore_index=True)

In [9]:
# Add 25% extra data to the dataset by bootstrapping (sampling with replacement)
df_T7 = df_base.copy()

increase_frac = 0.25
n_new = int(len(df_T7) * increase_frac)

bootstrap_sample = df_T7.sample(
    n=n_new,
    replace=True,
    random_state=RANDOM_STATE
)

df_T7 = pd.concat([df_T7, bootstrap_sample], ignore_index=True)

### **5. Extra column**

In [10]:
# Add an artificial extra column: AnnualMileage
df_T8 = df_base.copy()

mileage_values = np.array([7000, 10000, 12000, 15000, 20000, 99999])

df_T8["AnnualMileage"] = rng.choice(
    mileage_values,
    size=len(df_T8),
    replace=True
)

### **6. Feature noise**

In [11]:
# Inject noise (10% of SD) and clip to valid range.
df_T9 = df_base.copy()

bm_std = df_T9["BonusMalus"].std()
noise = rng.normal(loc=0.0, scale=0.1 * bm_std, size=len(df_T9))

df_T9["BonusMalus"] = df_T9["BonusMalus"] + noise
df_T9["BonusMalus"] = df_T9["BonusMalus"].clip(lower=50, upper=350)

In [12]:
# Inject noise (10% of SD) and clip to valid range.
df_T10 = df_base.copy()

bm_std = df_T10["DrivAge"].std()
noise = rng.normal(loc=0.0, scale=0.1 * bm_std, size=len(df_T10))

df_T10["DrivAge"] = df_T10["DrivAge"] + noise
df_T10["DrivAge"] = df_T10["DrivAge"].clip(lower=18, upper=85)

### **7. Label noise**

In [13]:
# Increase claims for a random 5% of policies.
df_T11 = df_base.copy()

mask = rng.random(len(df_T11)) < 0.05
df_T11.loc[mask, "ClaimNb"] += 1

In [14]:
# Increase claims for a random 20% of policies.
df_T12 = df_base.copy()

mask = rng.random(len(df_T12)) < 0.2
df_T12.loc[mask, "ClaimNb"] += 1

### **8. Distribution shift**

In [15]:
# Distribution shift (older vehicles over-represented)
df_T13 = df_base.copy()

old_vehicles = df_T13[df_T13["VehAge"] >= 10]
df_T13 = pd.concat(
    [df_T13, old_vehicles.sample(frac=0.5, random_state=RANDOM_STATE)],
    ignore_index=True
)

In [16]:
# Distribution shift (young drivers over-represented)
df_T14 = df_base.copy()

young_drivers = df_T14[df_T14["DrivAge"] <= 25]
df_T14 = pd.concat(
    [df_T14, young_drivers.sample(frac=0.5, random_state=RANDOM_STATE)],
    ignore_index=True
)

### **9. Fairness stress test**

In [17]:
# Increase claims only for young drivers
df_T15 = df_base.copy()

mask = df_T15["DrivAge"] <= 25
df_T15.loc[mask, "ClaimNb"] += 1

### **10. Counterfactual tests**

In [18]:
# Random permutation within the BonusMalus column
df_T16 = df_base.copy()

df_T16["BonusMalus"] = rng.permutation(df_T16["BonusMalus"].values)

In [19]:
# Random permutation within the VehPower column
df_T17 = df_base.copy()

df_T17["VehPower"] = rng.permutation(df_T17["VehPower"].values)

### **Export datasets**

In [20]:
# Compile all datasets into a dictionary for easy access
datasets = {
    "T1_missing_values": df_T1,
    "T2_missing_values": df_T2,
    "T3_missing_rows": df_T3,
    "T4_missing_rows": df_T4,
    "T5_missing_column": df_T5,
    "T6_data_increase": df_T6,
    "T7_data_increase": df_T7,
    "T8_extra_column": df_T8,
    "T9_feature_noise": df_T9,
    "T10_feature_noise": df_T10,
    "T11_label_noise": df_T11,
    "T12_label_noise": df_T12,
    "T13_distribution_shift": df_T13,
    "T14_distribution_shift": df_T14,
    "T15_fairness_test": df_T15,
    "T16_counterfactual": df_T16,
    "T17_counterfactual": df_T17,
}

In [21]:
# Export datasets to CSV files
#for name, d in datasets.items():
#    d.to_csv(f"C:/Users/bart_/Documents/git_repo/agentic_actuaries/data/raw/{name}.csv", index=False)

In [ ]:
def calculate_psi(base, new, bins=10):
    base = base.dropna()
    new = new.dropna()
    
    quantiles = np.linspace(0, 1, bins + 1)
    breakpoints = np.quantile(base, quantiles)
    
    base_counts, _ = np.histogram(base, bins=breakpoints)
    new_counts, _ = np.histogram(new, bins=breakpoints)
    
    base_perc = base_counts / len(base)
    new_perc = new_counts / len(new)
    
    base_perc = np.where(base_perc == 0, 1e-6, base_perc)
    new_perc = np.where(new_perc == 0, 1e-6, new_perc)
    
    psi = np.sum((new_perc - base_perc) * np.log(new_perc / base_perc))
    
    return psi

In [23]:
def correlation_shift(df_base, df_new):
    num_cols = df_base.select_dtypes(include=np.number).columns
    
    corr_base = df_base[num_cols].corr().values
    corr_new = df_new[num_cols].corr().values
    
    return np.linalg.norm(corr_base - corr_new)

In [ ]:
def compute_dataset_severity(df_base, df_new):
    
    severity = {}
    
    severity["row_change"] = abs(len(df_new) - len(df_base)) / len(df_base)
    severity["col_change"] = abs(df_new.shape[1] - df_base.shape[1]) / df_base.shape[1]
    
    missing_base = df_base.isna().sum().sum()
    missing_new = df_new.isna().sum().sum()
    total_cells = df_base.size
    severity["missingness"] = abs(missing_new - missing_base) / total_cells
    
    if "ClaimNb" in df_new.columns:
        mu_base = df_base["ClaimNb"].mean()
        mu_new = df_new["ClaimNb"].mean()
        sigma_base = df_base["ClaimNb"].std()
        severity["target_shift"] = abs(mu_new - mu_base) / sigma_base
    else:
        severity["target_shift"] = 0
    
    psi_features = []
    for col in ["DrivAge", "VehAge", "BonusMalus"]:
        if col in df_new.columns:
            psi_features.append(calculate_psi(df_base[col], df_new[col]))
    
    severity["avg_psi"] = np.mean(psi_features) if psi_features else 0
    
    try:
        severity["corr_shift"] = correlation_shift(df_base, df_new)
    except:
        severity["corr_shift"] = 0
    
    severity["composite_score"] = (
        severity["row_change"]
        + severity["col_change"]
        + severity["missingness"]
        + severity["target_shift"]
        + severity["avg_psi"]
        + severity["corr_shift"]
    )
    
    return severity

In [25]:
severity_results = {}

for name, d in datasets.items():
    severity_results[name] = compute_dataset_severity(df_base, d)

severity_df = pd.DataFrame(severity_results).T
print(severity_df.sort_values("composite_score", ascending=False))

                        row_change  col_change  missingness  target_shift  \
T12_label_noise           0.000000    0.000000     0.000000      0.830257   
T15_fairness_test         0.000000    0.000000     0.000000      0.238909   
T16_counterfactual        0.000000    0.000000     0.000000      0.000000   
T11_label_noise           0.000000    0.000000     0.000000      0.209278   
T7_data_increase          0.250000    0.000000     0.000000      0.000371   
T13_distribution_shift    0.165318    0.000000     0.000000      0.002711   
T17_counterfactual        0.000000    0.000000     0.000000      0.000000   
T4_missing_rows           0.147490    0.000000     0.000000      0.000812   
T14_distribution_shift    0.028684    0.000000     0.000000      0.002606   
T5_missing_column         0.000000    0.083333     0.000000      0.000000   
T8_extra_column           0.000000    0.083333     0.000000      0.000000   
T6_data_increase          0.049999    0.000000     0.000000      0.000141   

In [27]:
severity_df.to_csv("C:/Users/bart_/Documents/Thesis/results/dataset_severity_results.csv", index=True)  